In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 03 · Preprocesamiento — de las decisiones a código con pruebas

Cada decisión del EDA (`reports/01_eda.md`) y de la revisión de atípicos (`especificaciones/03-preprocesamiento.md`, sección 3b)
es una **función pequeña** en `src/proyecto/` con su prueba en `tests/`. Este notebook **solo orquesta**: llama a las funciones y
comprueba resultados.

| Capa | Dónde | Cuándo se aplica | Qué hace |
|---|---|---|---|
| Limpieza **sin aprendizaje** | `proyecto.data.cleaning.clean` | Antes de partir | Quita `Policy_Sales_Channel` (posible fuga), región a entero, texto a números, marca de prima mínima, imposibles a vacío |
| Partición | `proyecto.data.split` | — | 80/20 estratificado por `Response`, semilla 42; repetidos fuera **solo** en entrenamiento |
| Preprocesador **que aprende** | `proyecto.features.preprocessing.build_preprocessor` | Dentro del `Pipeline`, ajustado solo con entrenamiento | Recorte de la prima al p99,9 + `log1p`, categorías raras → "otros", imputación y escalado |

## 0. Cargar los datos crudos

In [2]:
import numpy as np
import pandas as pd

from proyecto.config import FEATURE_COLUMNS, ID_COL, MIN_PREMIUM, PROCESSED_DIR, ROOT, TARGET
from proyecto.data.cleaning import clean, find_raw_file, load_raw

raw_path = find_raw_file()
df = load_raw(raw_path)       # verifica que vengan exactamente las 12 columnas esperadas (sin datos personales)
print(f"{raw_path.relative_to(ROOT).as_posix()}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

data/raw/dataset.csv: 381,109 filas x 12 columnas


## 1. Limpieza (reglas sin aprendizaje)

In [3]:
clean_df = clean(df)
print(f"antes: {df.shape}  ->  después: {clean_df.shape}  (sale Policy_Sales_Channel, entra is_min_premium)")
clean_df.dtypes.to_frame("tipo").T

antes: (381109, 12)  ->  después: (381109, 12)  (sale Policy_Sales_Channel, entra is_min_premium)


,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Vintage,Response,is_min_premium
tipo,int64,Int64,float64,Int64,Int64,Int64,Int64,Int64,float64,float64,Int64,int64


## 2. Comprobaciones de la limpieza

Cada decisión se convierte en una **afirmación** que falla si no se cumple.

In [4]:
checks = {
    "no se perdieron filas": len(clean_df) == len(df),
    "mismos clientes, mismo orden": clean_df[ID_COL].equals(df[ID_COL]),
    "sin columnas de texto": all(pd.api.types.is_numeric_dtype(clean_df[c]) for c in clean_df.columns),
    "Policy_Sales_Channel eliminada (posible fuga)": "Policy_Sales_Channel" not in clean_df.columns,
    "código de región entero": str(clean_df["Region_Code"].dtype) == "Int64",
    "is_min_premium marca la prima mínima (64.877 clientes)": int(clean_df["is_min_premium"].sum()) == int(df["Annual_Premium"].eq(MIN_PREMIUM).sum()) == 64877,
    "prima conservada (no se borra la cola alta)": clean_df["Annual_Premium"].max() == df["Annual_Premium"].max(),
    "Vehicle_Age ordenada 0/1/2": set(clean_df["Vehicle_Age"].unique()) == {0, 1, 2},
    "sin valores imposibles en el dataset actual": int(clean_df.isna().sum().sum()) == 0,
    "datos crudos sin modificar": df["Gender"].isin(["Male", "Female"]).all(),
}
result = pd.Series(checks, name="se cumple")
assert result.all(), result[~result]
result.to_frame()

,se cumple
no se perdieron filas,True
"mismos clientes, mismo orden",True
sin columnas de texto,True
Policy_Sales_Channel eliminada (posible fuga),True
código de región entero,True
is_min_premium marca la prima mínima (64.877 clientes),True
prima conservada (no se borra la cola alta),True
Vehicle_Age ordenada 0/1/2,True
sin valores imposibles en el dataset actual,True
datos crudos sin modificar,True


## 3. Partición

Cada cliente aparece **una sola vez** (381.109 `id` distintos), así que partir por fila equivale a partir por cliente.
Se estratifica por `Response` para conservar el 12,26 % de interesados. Los perfiles repetidos se quitan **solo en entrenamiento**.

In [5]:
from proyecto.data.split import drop_duplicate_rows, split_train_test

train_full, test = split_train_test(clean_df)
train = drop_duplicate_rows(train_full)

assert set(train[ID_COL]).isdisjoint(test[ID_COL]), "hay clientes en los dos lados"
assert len(train_full) + len(test) == len(clean_df), "se perdieron filas al partir"
assert abs(train[TARGET].mean() - test[TARGET].mean()) < 0.001, "la estratificación no se cumplió"

pd.DataFrame({
    "filas": [len(train_full), len(train), len(test)],
    "tasa de interés": [train_full[TARGET].mean(), train[TARGET].mean(), test[TARGET].mean()],
}, index=["entrenamiento (antes de quitar repetidos)", "entrenamiento (final)", "prueba (cerrada)"]).round(4)

,filas,tasa de interés
entrenamiento (antes de quitar repetidos),304887,0.1226
entrenamiento (final),304238,0.1227
prueba (cerrada),76222,0.1226


In [6]:
print(f"repetidos quitados del entrenamiento: {len(train_full) - len(train)}")

repetidos quitados del entrenamiento: 649


**Mi unidad de análisis y por qué parto así:** una fila es un cliente; ningún cliente se repite, así que partir por fila es válido.
La estratificación garantiza la misma proporción de interesados en entrenamiento y prueba. La prueba queda **cerrada** hasta la etapa 7.

## 4. Atípicos y categorías raras: el preprocesador se ajusta solo con entrenamiento

Aquí se ve **qué aprendió** el preprocesador. No se usa la prueba.

In [7]:
from proyecto.features.preprocessing import build_preprocessor

pre = build_preprocessor().fit(train[FEATURE_COLUMNS])
cap = pre.named_transformers_["premium"].named_steps["cap_log"].cap_
above = train["Annual_Premium"].gt(cap)
print(f"Tope de Annual_Premium aprendido (p99,9 de entrenamiento): {cap:,.0f}")
print(f"Clientes de entrenamiento por encima del tope (se recortan, no se borran): {above.sum():,} ({above.mean():.2%})")

Tope de Annual_Premium aprendido (p99,9 de entrenamiento): 115,478
Clientes de entrenamiento por encima del tope (se recortan, no se borran): 305 (0.10%)


In [8]:
onehot = pre.named_transformers_["cat"].named_steps["onehot"]
for col, cats, infrequent in zip(["Region_Code"], onehot.categories_, onehot.infrequent_categories_):
    n_inf = 0 if infrequent is None else len(infrequent)
    share = 0 if infrequent is None else train[col].isin(infrequent).mean()
    print(f"{col}: {len(cats)} categorías -> {len(cats) - n_inf} con columna propia + 'otros' ({n_inf} categorías, {share:.2%} de las filas)")

Region_Code: 53 categorías -> 48 con columna propia + 'otros' (5 categorías, 0.75% de las filas)


In [9]:
X_train = pd.DataFrame(pre.transform(train[FEATURE_COLUMNS]), columns=pre.get_feature_names_out(), index=train.index)
assert not X_train.isna().any().any(), "la salida tiene vacíos"
print(f"matriz de entrenamiento lista para modelar: {X_train.shape[0]:,} filas x {X_train.shape[1]} columnas")
X_train.iloc[:, :9].describe().T[["mean", "std", "min", "max"]].round(2)

matriz de entrenamiento lista para modelar: 304,238 filas x 58 columnas


,mean,std,min,max
premium__Annual_Premium,-0.00,1.00,-2.14,1.64
num__Age,-0.00,1.00,-1.21,2.98
num__Vintage,0.00,1.00,-1.73,1.73
flags__Gender,0.54,0.50,0.00,1.00
flags__Driving_License,1.00,0.05,0.00,1.00
flags__Previously_Insured,0.46,0.50,0.00,1.00
flags__Vehicle_Damage,0.50,0.50,0.00,1.00
flags__Vehicle_Age,0.61,0.57,0.00,2.00
flags__is_min_premium,0.17,0.37,0.00,1.00


## 5. Guardar los datos de la etapa

Se guardan los datos **limpios y partidos** (sin transformar: el preprocesador se ajusta dentro del `Pipeline` de cada modelo en la
etapa 04). `data/processed/` no se versiona: se regenera con este notebook.

In [10]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)
for name in ["train.parquet", "test.parquet"]:
    back = pd.read_parquet(PROCESSED_DIR / name)
    print(f"data/processed/{name}: {back.shape[0]:,} filas x {back.shape[1]} columnas")

data/processed/train.parquet: 304,238 filas x 12 columnas
data/processed/test.parquet: 76,222 filas x 12 columnas


## 6. Las pruebas automáticas

Desde la raíz del repo (con el entorno activado):

```bash
pytest tests -q
```

`tests/test_cleaning.py` (una prueba por regla, caso normal y caso sucio), `tests/test_split.py` (clientes disjuntos, sin pérdida de
filas, estratificación, reproducible) y `tests/test_preprocessing.py` (tope aprendido solo de entrenamiento, categorías raras y nuevas,
salida sin vacíos).

## Entregable de esta etapa (hito H1)

- [x] La limpieza corre de principio a fin desde el repo.
- [x] La partición respeta mi unidad de análisis y está **verificada con una prueba**.
- [x] `pytest` en verde.
- [x] `data/raw/` (y `dataset.csv`) no cambió.
- [x] Declaré mi unidad de análisis por escrito.